# MOSAIC, Taken Apart: a Hands-On Primer
**Series B: Deep Learning for Molecules, Post 6c** &nbsp;|&nbsp; companion *primer* notebook

MOSAIC (Li, Sarkar, Lu, ..., Newhouse and Batista, *Nature* **651**, 107, 2026; code at github.com/batistagroup/MOSAIC) turns the open Llama-3.1-8B-Instruct language model into **2,498 specialists**, one for each Voronoi cell of a learned map of reaction space, and answers a query by consulting the nearest few. It generates full experimental procedures for reactions of new compounds, with a confidence estimate: the distance from the query to the specialist's cell centre.

> **What this notebook is, and is not.** The real system needs the commercial Pistachio reaction database, GPUs and an 8-billion-parameter model, none of which are available here. This primer rebuilds the **mechanisms** in NumPy on a *synthetic reaction universe*: a learned reaction metric, Voronoi cells with IVF-style search, specialists per cell, set-based matching, distance as confidence, the experiment-ranking rule, and the scale arithmetic of LoRA. It does **not** run a language model and **does not reproduce MOSAIC's accuracy**; all toy numbers are for the toy. The paper's numbers (from the ChemRxiv preprint, not peer reviewed) are typed in where we analyse them. To run the real framework, use the authors' repository (`Main_Control.ipynb`).

| Section | What you do |
|---|---|
| 1 | build a synthetic universe of reactions with difference fingerprints |
| 2 | train a kernel metric network; compare the learned metric with raw fingerprints |
| 3 | partition the space into Voronoi cells (k-means) and search it IVF-style |
| 4 | build specialists, route queries, and score them with the paper's set metric |
| 5 | **a critical look at the metric:** more experts means bigger predicted sets |
| 6 | distance to the cell centre as a confidence |
| 7 | analyse the paper's Table 1 |
| 8 | the experiment-ranking rule |
| 9 | the paper's yield-prediction fit lines |
| 10 | scale arithmetic: Llama-3.1-8B, LoRA, and search cost |

## 0. Setup

In [ ]:
import itertools, math
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. A synthetic universe of reactions
MOSAIC encodes a reaction as concatenated **RDKit and Morgan fingerprints** of the reactants and of the product, plus a **difference fingerprint** (product minus reactants) that records *what changed*. We cannot run RDKit here, so we simulate the structure of this input. A reaction has a hidden **class** (40 classes in 8 families; classes of a family share part of their change pattern, like related coupling reactions). The reactant fingerprint is random substrate noise; the product is the reactant with the class's pattern of bits switched on and off, each bit applied with probability 0.65. The feature vector is $[r,\,p,\,p-r]$ (768 numbers).

In [ ]:
D, C, FAM = 256, 40, 8; PER = C // FAM
rng_u = np.random.default_rng(0)
fam_on  = [rng_u.choice(D, 8, replace=False) for _ in range(FAM)]
cls_on  = [np.r_[fam_on[c // PER], rng_u.choice(D, 4, replace=False)] for c in range(C)]       # bits a reaction of class c switches on
cls_off = [rng_u.choice(D, 4, replace=False) for c in range(C)]                                  # bits it switches off

def make_reaction(c, rng, keep=0.65, density=0.12):
    r = (rng.random(D) < density).astype(np.float32)                                              # reactant fingerprint: substrate noise
    r[cls_off[c]] = np.maximum(r[cls_off[c]], (rng.random(4) < 0.7))                              # the reactant carries the bits that will be removed
    p = r.copy(); p[cls_on[c][rng.random(len(cls_on[c])) < keep]] = 1; p[cls_off[c][rng.random(4) < keep]] = 0
    return np.r_[r, p, p - r]                                                                    # [reactant, product, difference]

def dataset(n, rng, prior, keep=0.65):
    y = rng.choice(C, n, p=prior); return np.stack([make_reaction(c, rng, keep) for c in y]), y

prior = rng_u.dirichlet(np.ones(C) * 1.5)                                                         # classes of very different sizes
Xtr, ytr = dataset(8000, rng_u, prior); Xte, yte = dataset(2000, rng_u, prior)
print(f"{len(Xtr)} training and {len(Xte)} test reactions, {C} classes in {FAM} families, {Xtr.shape[1]} features")
print(f"class sizes in the training set: min {np.bincount(ytr, minlength=C).min()}, max {np.bincount(ytr, minlength=C).max()}")
print(f"a typical difference fingerprint has {int((Xtr[0, 2 * D:] != 0).sum())} nonzero entries out of {D}")

## 2. The kernel metric network (KMN)
The paper trains a network to classify reactions into **2,285 classes** and takes the **128-dimensional layer before the output** as the *Reaction-Specific Fingerprint* (RSFP). The Euclidean distance between RSFPs then plays the role of a similarity kernel (Eq. 1 of the paper). Here: a small network $768\to256\to128\to40$ trained with softmax cross-entropy.
**Stop and predict:** how much better than the raw fingerprints will a nearest-neighbour search be in RSFP space?

In [ ]:
def init_kmn(rng):
    s = lambda a, b: (rng.normal(size=(a, b)) * np.sqrt(2 / a)).astype(np.float32)
    return dict(W1=s(3 * D, 256), b1=np.zeros(256, np.float32), W2=s(256, 128), b2=np.zeros(128, np.float32), W3=s(128, C) * 0.5, b3=np.zeros(C, np.float32))

def forward(P, X):
    h = np.maximum(X @ P["W1"] + P["b1"], 0); z = np.tanh(h @ P["W2"] + P["b2"])                  # z is the 128-dim RSFP
    return h, z, z @ P["W3"] + P["b3"]

def train_kmn(P, X, y, epochs=12, bs=128, lr=2e-3, seed=1):
    r = np.random.default_rng(seed); m = {k: np.zeros_like(v) for k, v in P.items()}; v = {k: np.zeros_like(x) for k, x in P.items()}; t = 0
    for ep in range(epochs):
        idx = r.permutation(len(X))
        for i in range(0, len(X), bs):
            b = idx[i:i + bs]; xb, yb = X[b], y[b]; h, z, lg = forward(P, xb)
            p = np.exp(lg - lg.max(1, keepdims=True)); p /= p.sum(1, keepdims=True); g = p; g[np.arange(len(b)), yb] -= 1; g /= len(b)
            gpre2 = (g @ P["W3"].T) * (1 - z ** 2); gh = gpre2 @ P["W2"].T; gpre1 = gh * (h > 0)
            grads = dict(W1=xb.T @ gpre1, b1=gpre1.sum(0), W2=h.T @ gpre2, b2=gpre2.sum(0), W3=z.T @ g, b3=g.sum(0)); t += 1
            for k in P:
                m[k] = 0.9 * m[k] + 0.1 * grads[k]; v[k] = 0.999 * v[k] + 0.001 * grads[k] ** 2
                P[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
    return P

kmn = train_kmn(init_kmn(np.random.default_rng(2)), Xtr, ytr)
_, ztr, _ = forward(kmn, Xtr); _, zte, lgte = forward(kmn, Xte)
print(f"classification accuracy on test reactions: {(lgte.argmax(1) == yte).mean():.3f}")

def nn_accuracy(A, ya, B, yb):                                                  # 1-nearest-neighbour class accuracy
    d = (B ** 2).sum(1)[:, None] + (A ** 2).sum(1)[None, :] - 2 * B @ A.T; return (ya[d.argmin(1)] == yb).mean()

print("\n1-nearest-neighbour class accuracy (Euclidean distance):")
print(f"   raw [reactant, product, difference] : {nn_accuracy(Xtr, ytr, Xte, yte):.3f}")
print(f"   difference fingerprint only          : {nn_accuracy(Xtr[:, 2 * D:], ytr, Xte[:, 2 * D:], yte):.3f}")
print(f"   learned RSFP (128 numbers)           : {nn_accuracy(ztr, ytr, zte, yte):.3f}")

In [ ]:
# a 2-D view of the learned space (first two principal components), coloured by reaction family
Zc = ztr - ztr.mean(0); U, S, Vt = np.linalg.svd(Zc[:3000], full_matrices=False); pc = Zc @ Vt[:2].T
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
cm = plt.cm.tab10
for f in range(FAM):
    sel = (ytr // PER) == f; ax[0].scatter(pc[sel, 0], pc[sel, 1], s=3, color=cm(f), label=f"family {f}")
ax[0].set(xlabel="PC 1", ylabel="PC 2", title="(a) RSFP space: related reactions cluster"); ax[0].legend(fontsize=6, ncol=2, markerscale=3)
Xc = Xtr[:, 2 * D:] - Xtr[:, 2 * D:].mean(0); U2, S2, Vt2 = np.linalg.svd(Xc[:3000], full_matrices=False); pr = Xc @ Vt2[:2].T
for f in range(FAM):
    sel = (ytr // PER) == f; ax[1].scatter(pr[sel, 0], pr[sel, 1], s=3, color=cm(f))
ax[1].set(xlabel="PC 1", ylabel="PC 2", title="(b) The same view of the raw difference fingerprints")
plt.tight_layout(); plt.show()

## 3. Voronoi cells and IVF search
MOSAIC clusters all RSFPs with FAISS's inverted-file index (IVF), whose coarse quantizer is **k-means**: the 128-dimensional space is cut into **Voronoi cells** around cluster centres. Each cell becomes one specialist's domain (2,500 cells in the paper; 30 here). A query is encoded, compared with the centres, and only the closest cell(s) are searched in detail.

In [ ]:
def kmeans(Z, K, rng, iters=30):
    cen = Z[rng.choice(len(Z), K, replace=False)].copy()
    for _ in range(iters):
        d = (Z ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * Z @ cen.T; a = d.argmin(1)
        for k in range(K):
            if (a == k).any(): cen[k] = Z[a == k].mean(0)
    return cen, a

K = 30
cen, atr = kmeans(ztr, K, np.random.default_rng(3))
sizes = np.bincount(atr, minlength=K)
print(f"{K} Voronoi cells; reactions per cell: min {sizes.min()}, median {int(np.median(sizes))}, max {sizes.max()}; empty cells: {(sizes == 0).sum()}")
purity = np.array([np.bincount(ytr[atr == k], minlength=C).max() / sizes[k] for k in range(K)])
fam_pur = np.array([np.bincount(ytr[atr == k] // PER, minlength=FAM).max() / sizes[k] for k in range(K)])
print(f"share of the dominant CLASS in a cell: mean {purity.mean():.2f} (min {purity.min():.2f}); of the dominant FAMILY: mean {fam_pur.mean():.2f}")

k0 = int(np.argmin(purity)); comp = np.bincount(ytr[atr == k0], minlength=C) / sizes[k0]; top = np.argsort(comp)[::-1][:3]
print(f"\nthe most mixed cell ({k0}, {sizes[k0]} reactions): " + ", ".join(f"class {c} ({100 * comp[c]:.1f} %)" for c in top))
print("(the paper's Fig. 1e: expert 883 is 70.69 % chloro Buchwald-Hartwig, 17.22 % chloro SNAr, 0.26 % Goldberg-type: related reactions share a cell)")

In [ ]:
# IVF search: probe only the nprobe nearest cells instead of every reaction
def ivf_search(zq, nprobe):
    dc = (zq ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * zq @ cen.T; order = np.argsort(dc, 1)
    found, cost = [], []
    for i in range(len(zq)):
        cand = np.where(np.isin(atr, order[i, :nprobe]))[0]
        d = ((ztr[cand] - zq[i]) ** 2).sum(1); found.append(cand[d.argmin()]); cost.append(K + len(cand))
    return np.array(found), np.array(cost)

d_all = (zte ** 2).sum(1)[:, None] + (ztr ** 2).sum(1)[None, :] - 2 * zte @ ztr.T; exact_nn = d_all.argmin(1)
print(f"{'nprobe':>6} | {'recall of the true nearest neighbour':>37} | {'distance computations / brute force':>36}")
for npb in (1, 2, 3, 5):
    f, cost = ivf_search(zte, npb); print(f"{npb:>6} | {np.mean(f == exact_nn):>37.3f} | {cost.mean():>15.0f} / {len(ztr)}  ({100 * cost.mean() / len(ztr):.1f} %)")

## 4. Specialists, routing and the paper's set metric
We need something for each specialist to predict. We invent **conditions**: each class has three typical (reagent set, solvent) combinations with probabilities 0.55 / 0.30 / 0.15 (related classes share reagents); a reaction's true conditions are drawn from them, with 15 % chance of one extra reagent and 10 % chance of a random solvent. A toy "specialist" for a cell simply lists the condition sets most frequent among its training reactions (the analogue of the model's top beam).
The paper scores predicted against true sets of reagents or solvents with
$$D=1-\frac{|S_{\rm pred}\cap S_{\rm true}|}{|S_{\rm true}|}:$$
$D=0$ means the **true set is a subset of the predicted set** ("exact match"); $D<1$ means at least one true molecule was predicted ("partial match"). Predictions come from the closest expert (*one-shot*) or from up to three nearest experts (*multiple-shots*).

In [ ]:
NR, NS = 30, 10
rc = np.random.default_rng(10)
fam_reag = [rc.choice(NR, 3, replace=False) for _ in range(FAM)]
typical = [[(frozenset({int(rc.choice(fam_reag[c // PER])), int(rc.integers(NR))}), int(rc.integers(NS))) for _ in range(3)] for c in range(C)]

def true_conditions(c, rng):
    reag, sol = typical[c][rng.choice(3, p=[0.55, 0.30, 0.15])]; reag = set(reag)
    if rng.random() < 0.15: reag.add(int(rng.integers(NR)))                    # an extra additive
    if rng.random() < 0.10: sol = int(rng.integers(NS))                         # an unusual solvent
    return frozenset(reag), sol

rng_c = np.random.default_rng(11)
cond_tr = [true_conditions(c, rng_c) for c in ytr]
keep_te = np.where(np.random.default_rng(12).random(len(yte)) < 0.8, 0.65, 0.35)       # 20 % of test reactions have a weaker class signal (harder, less typical)
Xte2 = np.stack([make_reaction(c, np.random.default_rng(1000 + i), keep=k) for i, (c, k) in enumerate(zip(yte, keep_te))])
_, zte2, _ = forward(kmn, Xte2); cond_te = [true_conditions(c, rng_c) for c in yte]

experts = []                                                                    # per cell: condition sets ordered by frequency (the 'beam')
for k in range(K):
    cnt = Counter(cond_tr[i] for i in np.where(atr == k)[0]); experts.append([s for s, _ in cnt.most_common()])

dist = (zte2 ** 2).sum(1)[:, None] + (cen ** 2).sum(1)[None, :] - 2 * zte2 @ cen.T       # squared Euclidean distance to every centre, as FAISS returns
order = np.argsort(dist, 1)

def D_metric(pred, true): return 1 - len(pred & true) / len(true)

def predict(i, n_experts):                                                         # top-1 prediction of each of the n nearest experts, pooled
    reag, sol = set(), set()
    for k in order[i, :n_experts]:
        if experts[k]: reag |= set(experts[k][0][0]); sol.add(experts[k][0][1])
    return reag, sol

def evaluate(n_experts):
    out = {"reagent": [], "solvent": [], "both": []}; size = []
    for i in range(len(zte2)):
        pr, ps = predict(i, n_experts); tr, ts = cond_te[i]; size.append(len(pr) + len(ps))
        out["reagent"].append(D_metric(pr, set(tr))); out["solvent"].append(D_metric(ps, {ts}))
        out["both"].append(D_metric(pr | {("s", x) for x in ps}, set(tr) | {("s", ts)}))
    return {k: (100 * np.mean(np.array(v) == 0), 100 * np.mean(np.array(v) < 1)) for k, v in out.items()}, float(np.mean(size))

rows = []
for n, label in ((1, "one-shot (nearest expert)"), (3, "multiple-shots (3 nearest experts)")):
    res, _ = evaluate(n)
    rows.append((label, *[f"{res[k][0]:.1f} / {res[k][1]:.1f}" for k in ("reagent", "solvent", "both")]))
print("toy universe: exact / partial match rates (%)")
print(pd.DataFrame(rows, columns=["", "reagent", "solvent", "both"]).to_string(index=False))
print("\n(the paper's preprint reports, for real reactions: one-shot reagent 22.4 / 45.4, solvent 29.8 / 51.7, both 12.9 / 73.0; multiple-shots 43.0 / 76.0, 32.8 / 55.2, 28.9 / 94.8)")

## 5. A critical look at "exact match"
Because $D=0$ whenever the true set is *contained* in the prediction, a prediction that lists more molecules can only do better. Using up to three experts enlarges the predicted set, so part of the improvement from "one-shot" to "multiple-shots" is mechanical. The metric ignores **precision**. We measure both: how the exact-match rate, the size of the predicted set and the precision of the reagent prediction change as we pool more experts, and what a degenerate predictor that lists *every* reagent would score.
**Stop and predict:** what does the degenerate predictor score?

In [ ]:
prec_rows = []
for n in (1, 2, 3, 5, 8):
    ex, prec, size = [], [], []
    for i in range(len(zte2)):
        pr, _ = predict(i, n); tr = set(cond_te[i][0]); ex.append(D_metric(pr, tr) == 0); size.append(len(pr)); prec.append(len(pr & tr) / max(len(pr), 1))
    prec_rows.append((f"{n} expert(s)", 100 * np.mean(ex), np.mean(size), 100 * np.mean(prec)))
everything = set(range(NR)); ex_all = np.mean([D_metric(everything, set(c[0])) == 0 for c in cond_te])
prec_rows.append(("predict ALL 30 reagents", 100 * ex_all, float(NR), 100 * np.mean([len(set(c[0])) / NR for c in cond_te])))
print(pd.DataFrame(prec_rows, columns=["predictor", "exact-match rate (%)", "mean size of predicted reagent set", "precision (%)"]).round(1).to_string(index=False))
print("\nthe exact-match rate rises with every added expert, but so does the number of molecules the chemist has to choose among, and precision falls.")
print("The degenerate predictor is 'exact' 100 % of the time. A fair reading of Table 1 needs the size of the predicted sets, or precision, next to the match rates.")

## 6. Distance to the cell centre as confidence
MOSAIC's confidence is the **distance from the query's RSFP to the centroid** of the expert that answers: small means the query lies well inside the expert's domain. (FAISS returns *squared* Euclidean distances, which is presumably the scale of the paper's thresholds: below 50 high confidence, 100 to 200 moderate, above 200 only the broad reaction category.) Our toy distances have their own scale, so we look at the *shape* of the relationship.

In [ ]:
d1 = dist[np.arange(len(zte2)), order[:, 0]]
exact1 = np.array([D_metric(set(experts[order[i, 0]][0][0]) if experts[order[i, 0]] else set(), set(cond_te[i][0])) == 0 for i in range(len(zte2))], float)
edges = np.quantile(d1, np.linspace(0, 1, 6)); table = []
for a, b in zip(edges[:-1], edges[1:]):
    m = (d1 >= a) & (d1 <= b); table.append((f"{a:.1f} to {b:.1f}", int(m.sum()), 100 * exact1[m].mean()))
print(pd.DataFrame(table, columns=["squared distance to the nearest centre", "reactions", "one-shot exact reagent match (%)"]).round(1).to_string(index=False))
rho, _ = spearmanr(d1, exact1); print(f"\nSpearman correlation between distance and success: {rho:.2f}")
weak = keep_te < 0.5
print(f"mean squared distance: typical reactions {d1[~weak].mean():.1f}, reactions with a weak class signal {d1[weak].mean():.1f}")
print("Even the closest quintile succeeds only about half the time: the true conditions are genuinely multi-modal (0.55 / 0.30 / 0.15 here). A small distance means 'in the domain', not 'right'.")

plt.figure(figsize=(6.5, 3.8)); plt.bar(range(5), [t[2] for t in table], color=COLORS["blue"]); plt.xticks(range(5), ["closest", "2", "3", "4", "farthest"])
plt.xlabel("quintile of distance to the nearest centre"); plt.ylabel("one-shot exact reagent match (%)"); plt.tight_layout(); plt.show()

## 7. Analysing the paper's Table 1
Typed in from the ChemRxiv preprint (reagent and solvent prediction on held-out reactions; "both" treats reagents and solvents as one set). Exact means $D=0$, partial means $D<1$.

In [ ]:
t1 = pd.DataFrame({"reagent": [22.4, 45.4, 43.0, 76.0], "solvent": [29.8, 51.7, 32.8, 55.2], "both": [12.9, 73.0, 28.9, 94.8]},
                  index=pd.MultiIndex.from_product([["one-shot", "multiple-shots"], ["exact", "partial"]]))
print(t1.to_string())
g = (t1.loc["multiple-shots"] / t1.loc["one-shot"]).round(2); print("\nmultiple-shots / one-shot:\n", g.to_string())
fail1, failn = 100 - t1.loc[("one-shot", "partial"), "both"], 100 - t1.loc[("multiple-shots", "partial"), "both"]
print(f"\nreactions where NOT ONE correct reagent or solvent is predicted: {fail1:.1f} % (one-shot) -> {failn:.1f} % (multiple-shots), a factor {fail1 / failn:.1f} fewer")
print("consistency checks: partial >= exact everywhere:", bool((t1.xs('partial', level=1).values >= t1.xs('exact', level=1).values).all()),
      "| 'both' partial >= reagent partial and solvent partial:", bool((t1.xs('partial', level=1)['both'] >= t1.xs('partial', level=1)[['reagent', 'solvent']].max(axis=1)).all()))
print("solvents gain much less from extra experts than reagents (exact x1.10 vs x1.92): there are fewer plausible solvents, so one expert already covers most of them")

# a crude model: each of 3 experts covers the truth independently with the one-shot probability q.  Then 1 - (1 - q)^3 would be the multiple-shots rate.
# Solve 1 - (1 - q)^m = observed for m: the EFFECTIVE number of independent experts that the three nearest experts amount to.
eff = {}
for col in t1.columns:
    for kind in ("exact", "partial"):
        q, obs = t1.loc[("one-shot", kind), col] / 100, t1.loc[("multiple-shots", kind), col] / 100
        eff[(col, kind)] = (1 - (1 - q) ** 3, obs, math.log(1 - obs) / math.log(1 - q))
print(pd.DataFrame(eff, index=["independence prediction", "observed", "effective independent experts (of 3)"]).T.round(2).to_string())
print("reagents behave like about 2.2 independent experts, solvents like about 1.1: the three experts nearly always suggest the same solvent")

## 8. The experiment-ranking rule
With $N$ experts and expert $e$ offering $M_e$ predictions, the paper ranks the experiment (expert $e$, prediction $p$) by
$$R(e,p)=N\,(p-1)+e,$$
so that every expert's first prediction is tried before any expert's second.

In [ ]:
def ranking(M):                                                         # M[e-1] = number of predictions of expert e
    N = len(M); return sorted(((N * (p - 1) + e, e, p) for e in range(1, N + 1) for p in range(1, M[e - 1] + 1)))

M = [4, 2, 3]
rk = ranking(M)
print("N = 3 experts offering", M, "predictions:")
print(pd.DataFrame(rk, columns=["rank R", "expert e", "prediction p"]).to_string(index=False))
ranks = [r for r, _, _ in rk]
print("\nranks unique:", len(set(ranks)) == len(ranks), "| all first predictions before any second prediction:",
      max(r for r, e, p in rk if p == 1) < min(r for r, e, p in rk if p == 2), "| ranks are dense (no gaps):", ranks == list(range(1, len(ranks) + 1)))
print("expert 2 has run out after two predictions, so ranks such as 8 and 11 never occur: the ranks order the experiments but are not consecutive")

## 9. The yield-prediction fit lines
The model writes the yield as text. To evaluate it, predictions are binned into ten intervals of 10 percentage points, and the figure of the preprint plots the median predicted yield of each bin against the average true yield of its members, with fit lines labelled $y=0.61x+24.99$ and $y=0.61x+24.93$ (the two panels: classes capped at 20 examples, and unconstrained).
**Stop and predict:** at which predicted yield does the fit say prediction and truth agree?

In [ ]:
a, b = 0.61, 24.99
fixed_point = b / (1 - a)
print(f"y = {a} x + {b} crosses y = x at x = {fixed_point:.1f} %: below it the model is too pessimistic on average, above it too optimistic")
centres = np.arange(5, 100, 10)
print(pd.DataFrame({"predicted yield, bin centre (%)": centres, "average true yield from the fit (%)": a * centres + b}).round(1).to_string(index=False))
print(f"\nthe slope {a} < 1 means the predictions are informative but over-dispersed: a predicted spread of 90 points corresponds to about {a * 90:.0f} true points.")
print(f"a linear recalibration x -> (y - {b}) / {a} would remove this compression (it cannot remove the scatter within each bin, which the preprint shows only in a figure)")

## 10. Scale arithmetic: Llama-3.1-8B, LoRA and search cost
Each expert is the Llama-3.1-8B-Instruct model plus a **LoRA adapter**: for a weight matrix $W\in\mathbb R^{d_{\rm out}\times d_{\rm in}}$, instead of retraining $W$ one learns $W'=W+\tfrac\alpha r BA$ with $B\in\mathbb R^{d_{\rm out}\times r}$, $A\in\mathbb R^{r\times d_{\rm in}}$ and rank $r\ll d$, i.e. $r\,(d_{\rm in}+d_{\rm out})$ parameters. **The paper does not state the rank or the target layers** (they are in the authors' training scripts, which we did not read), so the adapter sizes below are an *illustration*, not MOSAIC's configuration.

In [ ]:
# Llama-3.1-8B architecture (public model card values)
h, layers, inter, heads, kv_heads, vocab = 4096, 32, 14336, 32, 8, 128_256
head_dim = h // heads; kv = kv_heads * head_dim
attn = h * h + h * kv + h * kv + h * h                              # q, k, v, o projections
mlp = 3 * h * inter                                                 # gate, up, down projections
base_params = vocab * h * 2 + layers * (attn + mlp + 2 * h) + h     # embedding, output layer, blocks, norms
print(f"Llama-3.1-8B parameters from the architecture: {base_params / 1e9:.2f} billion ({2 * base_params / 1e9:.1f} GB in 16-bit)")

shapes = {"q": (h, h), "k": (h, kv), "v": (h, kv), "o": (h, h), "gate": (h, inter), "up": (h, inter), "down": (inter, h)}
def lora_params(r, targets):
    return layers * sum(r * (shapes[t][0] + shapes[t][1]) for t in targets)

N_EXPERTS = 2498
rows = []
for r, targets, label in ((8, "qkvo", "r = 8, attention only"), (16, "qkvo", "r = 16, attention only"), (16, ("q", "k", "v", "o", "gate", "up", "down"), "r = 16, all linear layers")):
    p = lora_params(r, targets); gb = 2 * p / 1e6
    rows.append((label, f"{p / 1e6:.1f} M", f"{100 * p / base_params:.2f} %", f"{gb:.0f} MB", f"{N_EXPERTS * gb / 1e3:.0f} GB"))
print(pd.DataFrame(rows, columns=["illustrative adapter", "parameters", "share of the base model", "size (16-bit)", f"{N_EXPERTS} adapters"]).to_string(index=False))
print(f"\nversus {N_EXPERTS} full copies of the model: {N_EXPERTS * 2 * base_params / 1e12:.0f} TB. Experts differ only by their adapter, so serving one means loading ~{2 * lora_params(16, ('q','k','v','o','gate','up','down')) / 1e6:.0f} MB, not 16 GB.")

In [ ]:
# training data per expert and the cost of searching the map: a rough, DERIVED estimate
# the preprint says its Reaxys/SciFinder-scale target (>100 million records) is 'nearly 100 times' the current training data
N_train = 100e6 / 100; cells = 2500
print(f"training reactions ~ {N_train / 1e6:.0f} million -> about {N_train / cells:.0f} reactions per Voronoi cell on average (cells are very unequal in size)")
for nprobe in (1, 3):
    ivf = cells + nprobe * N_train / cells
    print(f"IVF search with nprobe = {nprobe}: ~{ivf:,.0f} distance computations vs {N_train:,.0f} for brute force ({N_train / ivf:,.0f}x fewer)")
print("(the toy of Section 3 showed the same trade-off: recall of the exact nearest neighbour versus the fraction of the data scanned)")

## 11. Your turn
**Exercise 1 (metric).** For a true reagent set $\{A,B\}$ and a predicted set $\{A,C,D\}$, compute $D$. Is it an exact match? What if the prediction is $\{A,B,C,D,E\}$?

**Exercise 2 (cells).** Change `K` in Section 3 from 30 to 10 and to 100. How do the class purity, the number of reactions per cell, and the IVF recall at `nprobe=1` change? Which choice corresponds to the paper's design (about as many cells as reaction classes)?

**Exercise 3 (precision).** Define an F1-style score from the precision and the exact-match rate. At how many pooled experts is it highest in the toy?

**Exercise 4 (confidence).** Give each toy test reaction a "confidence" from its distance (for example, the three terciles). Plot the exact-match rate against it and decide at which distance you would stop trusting the expert.

**Exercise 5 (ranking).** For $N=4$ experts with $M=(3,3,1,2)$ predictions, list the first eight experiments in order. How many experiments are there in total?

**Exercise 6 (storage).** If the real adapters used rank 64 on all linear layers, how large would one adapter and the whole collection be?

In [ ]:
# Starters
# Exercise 1
print("D({A,B} vs {A,C,D}) =", round(D_metric({"A", "C", "D"}, {"A", "B"}), 2), "| D({A,B} vs {A,B,C,D,E}) =", D_metric({"A", "B", "C", "D", "E"}, {"A", "B"}))

# Exercise 5
rk4 = ranking([3, 3, 1, 2]); print("first eight experiments (rank, expert, prediction):", rk4[:8], "| total:", len(rk4))

# Exercise 6
p64 = lora_params(64, ("q", "k", "v", "o", "gate", "up", "down")); print(f"rank 64, all linear layers: {p64 / 1e6:.0f} M parameters = {2 * p64 / 1e6:.0f} MB per adapter, {N_EXPERTS * 2 * p64 / 1e12:.2f} TB for {N_EXPERTS}")

## 12. Self-check

In [ ]:
assert nn_accuracy(ztr, ytr, zte, yte) > nn_accuracy(Xtr, ytr, Xte, yte) + 0.1 and (lgte.argmax(1) == yte).mean() > 0.95
assert sizes.sum() == len(ztr) and fam_pur.mean() > 0.9
assert np.mean(ivf_search(zte, 1)[0] == exact_nn) < np.mean(ivf_search(zte, 5)[0] == exact_nn) and ivf_search(zte, 3)[1].mean() < 0.5 * len(ztr)
r1, r3 = evaluate(1)[0], evaluate(3)[0]
assert r3["reagent"][0] > r1["reagent"][0] and r3["both"][1] > r1["both"][1] and r3["reagent"][1] > r1["reagent"][1]
assert abs(D_metric({"A", "C", "D"}, {"A", "B"}) - 0.5) < 1e-12 and D_metric({"A", "B", "C"}, {"A", "B"}) == 0
assert ex_all == 1.0 and prec_rows[-1][3] < 30 and prec_rows[0][3] > prec_rows[-2][3]
assert table[-1][2] < table[0][2] - 15 and rho < -0.1
assert abs(fail1 - 27.0) < 1e-9 and abs(failn - 5.2) < 1e-9 and abs(g.loc["exact", "reagent"] - 1.92) < 1e-9 and abs(g.loc["exact", "solvent"] - 1.10) < 1e-9
assert abs(eff[("reagent", "exact")][2] - 2.22) < 0.02 and abs(eff[("solvent", "exact")][2] - 1.12) < 0.02
assert ranks == sorted(ranks) and len(set(ranks)) == len(ranks) and 8 not in ranks and 11 not in ranks
assert abs(fixed_point - 64.08) < 0.01 and abs(base_params / 1e9 - 8.03) < 0.01
print("All checks passed: you reproduced the numbers in the post.")